In [31]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import qiskit

Mermin-Peres Magic Square Game
Alice and Bob place a number 0 or 1 in a 3x3 square grid, The House choose which row Alice will be filling in (Index 0-2), the same for Bob but for column (Index 0-2). This selection is unknown to the pair prior to the game, ie. random. The rules state,
1. Alice selected row must be filled with 0 or 1 such that its sum is even (sum(x_i) mod 2 = 0)
2. Bob selected row must be filled with 0 or 1 such that its sum is odd (sum(x_j) mod 2 = 1)
3. Both will win if the intersected square has the same value (0 or 1)

Insight
1. Alice eligible row selection are {000, 011, 101, 110}
2. Bob eligible column selection are {001, 010, 100, 111}

In [2]:
def eligible_array(array, name):
    if name == "Alice":
        return (sum(array) % 2 == 0) and (len(array) == 3)
    elif name == "Bob":
        return (sum(array) % 2 == 1) and (len(array) == 3)
    else:
        print("Invalid name")
        return False

In [3]:
alice_row_eligible = [[0,0,0], [0,1,1], [1,0,1], [1,1,0]]
bob_col_eligible = [[0,0,1], [0,1,0], [1,0,0], [1,1,1]]

In [4]:
falty = False
for i in range(4):
    if not eligible_array(alice_row_eligible[i], "Alice"):
        print(f"Row {i} for Alice is not eligible")
        falty = True
        break
    if not eligible_array(bob_col_eligible[i], "Bob"):
        print(f"Column {i} for Bob is not eligible")
        falty = True
        break
if not falty:
    print("All rows and columns are eligible")

All rows and columns are eligible


House selecting their row/col index to fill in.

In [5]:
alice_row_index = random.randint(0, 2)
bob_col_index = random.randint(0, 2)

In [6]:
print(f"Alice's row to fill: {alice_row_index}, Bob's column to fill: {bob_col_index}")

Alice's row to fill: 2, Bob's column to fill: 2


Strategy 1: Random Selection 

In [7]:
alices_choice = random.choice(alice_row_eligible)
bobs_choice = random.choice(bob_col_eligible)

print(f"Alice's choice: {alices_choice}, Bob's choice: {bobs_choice}")

Alice's choice: [1, 0, 1], Bob's choice: [0, 0, 1]


In [8]:
def verify_game(alice_row_index, bob_col_index, alice_row, bob_col):
    return alice_row[bob_col_index] == bob_col[alice_row_index]

In [9]:
def square_game_matrix(alice_row_index, bob_col_index, alice_row, bob_col):
    matrix = np.full((3, 3), np.nan)
    matrix[alice_row_index, :] = alice_row
    matrix[:, bob_col_index] = bob_col
    matrix[alice_row_index, bob_col_index] = alice_row[alice_row_index] if alice_row[bob_col_index] == bob_col[alice_row_index] else np.nan
    return matrix

In [10]:
matrix = square_game_matrix(alice_row_index, bob_col_index, alices_choice, bobs_choice)
is_won = verify_game(alice_row_index, bob_col_index, alices_choice, bobs_choice)

print("Game matrix:\n", matrix)
print("Game won:", is_won)

Game matrix:
 [[nan nan  0.]
 [nan nan  0.]
 [ 1.  0.  1.]]
Game won: True


Random Stratergy

In [11]:
num_simulations = 10**6

def simulate_game(num_simulations):
    wins = 0
    for _ in range(num_simulations):
        alice_row_index = random.randint(0, 2)
        bob_col_index = random.randint(0, 2)
        alices_choice = random.choice(alice_row_eligible)
        bobs_choice = random.choice(bob_col_eligible)
        is_won = verify_game(alice_row_index, bob_col_index, alices_choice, bobs_choice)
        if is_won:
            wins += 1
    return wins / num_simulations

In [12]:
print('Probability of winning:', simulate_game(num_simulations))

Probability of winning: 0.499666


In [13]:
all_strartegies = []
for i in range(4):
    for j in range(4):
        all_strartegies.append((alice_row_eligible[i], bob_col_eligible[j]))

In [14]:
print("All static strategies: \n", all_strartegies)

All static strategies: 
 [([0, 0, 0], [0, 0, 1]), ([0, 0, 0], [0, 1, 0]), ([0, 0, 0], [1, 0, 0]), ([0, 0, 0], [1, 1, 1]), ([0, 1, 1], [0, 0, 1]), ([0, 1, 1], [0, 1, 0]), ([0, 1, 1], [1, 0, 0]), ([0, 1, 1], [1, 1, 1]), ([1, 0, 1], [0, 0, 1]), ([1, 0, 1], [0, 1, 0]), ([1, 0, 1], [1, 0, 0]), ([1, 0, 1], [1, 1, 1]), ([1, 1, 0], [0, 0, 1]), ([1, 1, 0], [0, 1, 0]), ([1, 1, 0], [1, 0, 0]), ([1, 1, 0], [1, 1, 1])]


Static Selection Stratergy

In [15]:
num_simulations = 10**6

for strategy in all_strartegies:
    alices_choice, bobs_choice = strategy
    wins = 0
    for _ in range(num_simulations):
        # For Alice
        alice_row_index = random.randint(0, 2)
        # For Bob
        bob_col_index = random.randint(0, 2)

        is_won = verify_game(alice_row_index, bob_col_index, alices_choice, bobs_choice)
        if is_won:
            wins += 1
    print(f"Probability of winning with strategy {strategy}: {wins / num_simulations}")

Probability of winning with strategy ([0, 0, 0], [0, 0, 1]): 0.666607
Probability of winning with strategy ([0, 0, 0], [0, 1, 0]): 0.666587
Probability of winning with strategy ([0, 0, 0], [1, 0, 0]): 0.666798
Probability of winning with strategy ([0, 0, 0], [1, 1, 1]): 0.0
Probability of winning with strategy ([0, 1, 1], [0, 0, 1]): 0.444431
Probability of winning with strategy ([0, 1, 1], [0, 1, 0]): 0.444102
Probability of winning with strategy ([0, 1, 1], [1, 0, 0]): 0.44458
Probability of winning with strategy ([0, 1, 1], [1, 1, 1]): 0.667003
Probability of winning with strategy ([1, 0, 1], [0, 0, 1]): 0.444673
Probability of winning with strategy ([1, 0, 1], [0, 1, 0]): 0.445553
Probability of winning with strategy ([1, 0, 1], [1, 0, 0]): 0.445079
Probability of winning with strategy ([1, 0, 1], [1, 1, 1]): 0.666516
Probability of winning with strategy ([1, 1, 0], [0, 0, 1]): 0.444749
Probability of winning with strategy ([1, 1, 0], [0, 1, 0]): 0.444426
Probability of winning wit

In [16]:
print("ALice [0, 0, 0]:", eligible_array([0, 0, 0], "Alice"))
print("Alice [1, 1, 0]:", eligible_array([1, 1, 0], "Alice"))
print("Bob [0, 0, 1]:", eligible_array([0, 0, 1], "Bob"))

ALice [0, 0, 0]: True
Alice [1, 1, 0]: True
Bob [0, 0, 1]: True


Best stratergy

In [17]:
num_simulations = 10**6

wins = 0
house_choice_failure = set()
for _ in range(num_simulations):
    alice_row_index = random.randint(0, 2)
    bob_col_index = random.randint(0, 2)

    # For Alice:
    alices_choice = [0, 0, 0] if alice_row_index != 2 else [1, 1, 0]
    # For Bob:
    bobs_choice = [0, 0, 1]

    is_won = verify_game(alice_row_index, bob_col_index, alices_choice, bobs_choice)
    if is_won:
        wins += 1
    else:
        house_choice_failure.add((alice_row_index, bob_col_index))

print(f"Probability of winning with this strategy: {wins / num_simulations}")
print("House choice failures:", house_choice_failure)

Probability of winning with this strategy: 0.889112
House choice failures: {(2, 2)}


In [18]:
v = np.array([1/2, -1/2, -1/2, 1/2])
v_dagger = v.conjugate().T

print("Vector v:", v)
print("Conjugate of v:", v_dagger)

print("Result of v * v_dagger:", np.dot(v, v_dagger))

Vector v: [ 0.5 -0.5 -0.5  0.5]
Conjugate of v: [ 0.5 -0.5 -0.5  0.5]
Result of v * v_dagger: 1.0


In [19]:
theta = np.pi / 4

Ry_plus = np.array([[np.cos(theta / 2), -np.sin(theta / 2)],
                    [np.sin(theta / 2), np.cos(theta / 2)]])

Ry_minus = np.array([[np.cos(-theta / 2), -np.sin(-theta / 2)],
                    [np.sin(-theta / 2), np.cos(-theta / 2)]])

H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)

In [20]:
Ry_minus_H = np.kron(Ry_minus, H)
Ry_plus_H = np.kron(Ry_plus, H)

In [21]:
print("cos theta:", np.cos(theta / 2))
print("sin theta:", np.sin(theta / 2))

cos theta: 0.9238795325112867
sin theta: 0.3826834323650898


In [22]:
print("Hadamard matrix H:\n", H)
print("Ry_plus:\n", Ry_plus)
print("Ry_minus:\n", Ry_minus)
print("Ry_minus_H:\n", Ry_minus_H)
print("Ry_plus_H:\n", Ry_plus_H)

Hadamard matrix H:
 [[ 0.70710678  0.70710678]
 [ 0.70710678 -0.70710678]]
Ry_plus:
 [[ 0.92387953 -0.38268343]
 [ 0.38268343  0.92387953]]
Ry_minus:
 [[ 0.92387953  0.38268343]
 [-0.38268343  0.92387953]]
Ry_minus_H:
 [[ 0.65328148  0.65328148  0.27059805  0.27059805]
 [ 0.65328148 -0.65328148  0.27059805 -0.27059805]
 [-0.27059805 -0.27059805  0.65328148  0.65328148]
 [-0.27059805  0.27059805  0.65328148 -0.65328148]]
Ry_plus_H:
 [[ 0.65328148  0.65328148 -0.27059805 -0.27059805]
 [ 0.65328148 -0.65328148 -0.27059805  0.27059805]
 [ 0.27059805  0.27059805  0.65328148  0.65328148]
 [ 0.27059805 -0.27059805  0.65328148 -0.65328148]]


In [23]:
result = np.dot(Ry_minus_H, np.array([1, 0, 0, 1]) / np.sqrt(2))
print(result)

[ 0.65328148  0.27059805  0.27059805 -0.65328148]


In [24]:
print("probability of measuring 0 or 3:", np.abs(result[0])**2 + np.abs(result[3])**2)

probability of measuring 0 or 3: 0.8535533905932733


In [25]:
print(np.abs(result[0])**2 + np.abs(result[3])**2 - (np.cos(theta / 2))**2)

-4.440892098500626e-16


In [26]:
r = np.sqrt(2)
theta = -np.pi / 4
z = r * np.exp(1j * theta)
A = np.array([[0, 1+1j], [z, 0]])
print("Matrix A:\n", A)

Matrix A:
 [[0.+0.j 1.+1.j]
 [1.-1.j 0.+0.j]]


In [27]:
eigenvalues, eigenvectors = np.linalg.eig(A)
print("Eigenvalues:", eigenvalues)
print("Eigenvectors:\n", eigenvectors)

Eigenvalues: [ 1.41421356+0.j -1.41421356+0.j]
Eigenvectors:
 [[ 0.70710678+0.j  -0.5       -0.5j]
 [ 0.5       -0.5j  0.70710678+0.j ]]


In [28]:
import sympy as sp

# Define symbolic matrix
A = sp.Matrix([[0, 1+sp.I], [sp.sqrt(2)*sp.exp(-sp.I*sp.pi/4), 0]])
print("Symbolic Matrix A:")
sp.pprint(A)

# Find eigenvalues symbolically
eigenvals = A.eigenvals()
eigenvects = A.eigenvects()

print("Symbolic Eigenvalues:")
for val in eigenvals:
    sp.pprint(val)

print("Symbolic Eigenvectors:")
for val, mult, vects in eigenvects:
    print(f"Eigenvalue: {val}")
    for vect in vects:
        sp.pprint(vect)

Symbolic Matrix A:
⎡    0      1 + ⅈ⎤
⎢                ⎥
⎢    -ⅈ⋅π        ⎥
⎢    ─────       ⎥
⎢      4         ⎥
⎣√2⋅ℯ         0  ⎦
Symbolic Eigenvalues:
          ________________
 4 ___   ╱ 4 ____         
-╲╱ 2 ⋅╲╱  ╲╱ -1 ⋅(1 - ⅈ) 
         ________________
4 ___   ╱ 4 ____         
╲╱ 2 ⋅╲╱  ╲╱ -1 ⋅(1 - ⅈ) 
Symbolic Eigenvectors:
Eigenvalue: -sqrt(2)
⎡  √2   √2⋅ⅈ⎤
⎢- ── - ────⎥
⎢  2     2  ⎥
⎢           ⎥
⎣     1     ⎦
Eigenvalue: sqrt(2)
⎡√2   √2⋅ⅈ⎤
⎢── + ────⎥
⎢2     2  ⎥
⎢         ⎥
⎣    1    ⎦


In [35]:
from qiskit.quantum_info import Pauli, Operator, SparsePauliOp

X = Pauli('X')
Y = Pauli('Y')
Z = Pauli('Z')
I = Pauli('I')

print("Pauli X matrix:")
print(X.to_matrix())
print("\nPauli Y matrix:")
print(Y.to_matrix())
print("\nPauli Z matrix:")
print(Z.to_matrix())
print("\nIdentity matrix:")
print(I.to_matrix())

Pauli X matrix:
[[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]

Pauli Y matrix:
[[0.+0.j 0.-1.j]
 [0.+1.j 0.+0.j]]

Pauli Z matrix:
[[ 1.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j]]

Identity matrix:
[[1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j]]


In [41]:
pauli_string = SparsePauliOp.from_list([("X", 1), ("Y", 1), ("Z", 1)])
print("\nSparse Pauli operators:")
print(pauli_string)
print(pauli_string.to_matrix())


Sparse Pauli operators:
SparsePauliOp(['X', 'Y', 'Z'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j])
[[ 1.+0.j  1.-1.j]
 [ 1.+1.j -1.+0.j]]


In [53]:
# Multi-qubit Pauli operators
XX = SparsePauliOp.from_list([("XX", 1)])
YY = SparsePauliOp.from_list([("YY", 1)])
ZZ = SparsePauliOp.from_list([("ZZ", 1)])

YZ = SparsePauliOp.from_list([("YZ", 1)])
ZX = SparsePauliOp.from_list([("ZX", 1)])
XY = SparsePauliOp.from_list([("XY", 1)])

ZY = SparsePauliOp.from_list([("ZY", 1)])
XZ = SparsePauliOp.from_list([("XZ", 1)])
YX = SparsePauliOp.from_list([("YX", 1)])

II = SparsePauliOp.from_list([("II", 1)])

In [54]:
print(XX)
print(XX.to_matrix())
print("\n")
print(YZ)
print(YZ.to_matrix())
print("\n")
print(ZY)
print(ZY.to_matrix())

SparsePauliOp(['XX'],
              coeffs=[1.+0.j])
[[0.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j]]


SparsePauliOp(['YZ'],
              coeffs=[1.+0.j])
[[0.+0.j 0.+0.j 0.-1.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+1.j]
 [0.+1.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.-1.j 0.+0.j 0.+0.j]]


SparsePauliOp(['ZY'],
              coeffs=[1.+0.j])
[[0.+0.j 0.-1.j 0.+0.j 0.+0.j]
 [0.+1.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+1.j]
 [0.+0.j 0.+0.j 0.-1.j 0.+0.j]]


In [58]:
print(XX@YZ@ZY)
print(YY@ZX@XZ)
print(ZZ@XY@YX)

SparsePauliOp(['II'],
              coeffs=[1.+0.j])
SparsePauliOp(['II'],
              coeffs=[1.+0.j])
SparsePauliOp(['II'],
              coeffs=[1.+0.j])


In [60]:
print(XX@YY@ZZ)
print(YZ@ZX@XY)
print(ZY@XZ@YX)

SparsePauliOp(['II'],
              coeffs=[-1.+0.j])
SparsePauliOp(['II'],
              coeffs=[-1.+0.j])
SparsePauliOp(['II'],
              coeffs=[-1.+0.j])
